# Public ASX OHLCV Parquet Demo

This notebook demonstrates that the processed parquet file is published on GitHub, includes preprocessing outputs, and is directly usable for analysis.

In [ ]:
import pandas as pd

In [ ]:
PARQUET_URL = "https://raw.githubusercontent.com/Marek-Czarnecki/data-analytics-capstone-public/main/data/processed/market_ohlcv_daily_v2/exchange%3DASX/asx_ohlcv_panel_clean.parquet"

print(f"PARQUET_URL={PARQUET_URL}")

In [ ]:
df = pd.read_parquet(PARQUET_URL)

print(df.shape)
df.head()

In [ ]:
required_columns = [
    "trade_date",
    "ticker",
    "open",
    "high",
    "low",
    "close",
    "ohlc_valid_flag",
    "ohlc_issue_type",
    "open_clean",
    "high_clean",
    "low_clean",
    "close_clean",
    "daily_return",
]

missing = [column for column in required_columns if column not in df.columns]
assert not missing, f"Missing expected preprocessing columns: {missing}"

summary = {
    "rows": len(df),
    "tickers": df["ticker"].nunique(),
    "date_min": df["trade_date"].min(),
    "date_max": df["trade_date"].max(),
    "invalid_ohlc_rows": int((~df["ohlc_valid_flag"]).sum()),
    "issue_types": df["ohlc_issue_type"].value_counts().to_dict(),
}

summary

In [ ]:
latest_close = (
    df.sort_values(["ticker", "trade_date"])
    .groupby("ticker", as_index=False)
    .tail(1)
    .loc[:, ["ticker", "trade_date", "close_clean", "daily_return"]]
    .sort_values("ticker")
)

latest_close.head(10)